In [31]:
from datasets import load_dataset
from collections import defaultdict
from sentence_transformers import SentenceTransformer
import numpy as np
from tqdm import tqdm
import faiss
import random

In [ ]:
# need to load in the dataset
dataset = load_dataset("microsoft/ms_marco", "v2.1", split="train[:1000]")
print(len(dataset))

In [ ]:
# a little bit of data exploration
example = dataset[0]
print(example.keys())

print("QUERY:", example["query"])
print("ANSWERS:", example["answers"])

passages = example["passages"]["passage_text"]
labels = example["passages"]["is_selected"]

print("NUM PASSAGES:", len(passages))
print("is_selected:", labels)

for i in range(3):
    print(f"\n--- Passage {i} (label={labels[i]}) ---")
    print(passages[i])

In [ ]:
corpus = {}
queries = {}
qrels = defaultdict(list)

In [ ]:
next_passage_id = 0
for example in dataset:
    qid = example["query_id"]
    query_text = example["query"]
    answers = example["answers"]
    
    queries[qid] = {
        "text": query_text,
        "answers": answers,
    }
    
    passage_texts = example["passages"]["passage_text"]
    labels = example["passages"]["is_selected"]
    
    for p_text, label in zip(passage_texts, labels):
        pid = next_passage_id
        corpus[pid] = p_text
        
        if label == 1:
            qrels[qid].append(pid)
            
        next_passage_id += 1
    

In [ ]:
print("Num queries:", len(queries))
print("Num passages:", len(corpus))

# pick an arbitrary query_id
some_qid = next(iter(queries.keys()))
print("\nExample query_id:", some_qid)
print("Query:", queries[some_qid]["text"])
print("Answers:", queries[some_qid]["answers"])
print("Relevant passage_ids:", qrels[some_qid])

for pid in qrels[some_qid]:
    print(f"\n--- Relevant passage {pid} ---")
    print(corpus[pid])

In [ ]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(model_name)

# making sure this is working right
embedder.encode("hello world").shape

In [ ]:
num_passages = len(corpus)
passage_texts = [corpus[pid] for pid in range(num_passages)]

In [ ]:
# encode all the passages in batches
batch_size = 64

all_embeddings = []

for i in tqdm(range(0, num_passages, batch_size)):
    batch_texts = passage_texts[i : i + batch_size]
    batch_emb = embedder.encode(
        batch_texts, 
        convert_to_numpy=True, 
        normalize_embeddings=True
    )
    all_embeddings.append(batch_emb)

passage_embeddings = np.vstack(all_embeddings)

print(passage_embeddings.shape) # apparently i need should expect something like (num_passages, embedding_dim)

In [ ]:
dim = passage_embeddings.shape[1]

index = faiss.IndexFlatIP(dim)
index.add(passage_embeddings)

print("Index size:", index.ntotal)  # should be num_passages

In [ ]:
some_qid = next(iter(queries.keys()))
query_text = queries[some_qid]["text"]
print("Query:", query_text)
print("Answers:", queries[some_qid]["answers"])
print("Relevant passage_ids:", qrels[some_qid])

In [ ]:
query_vec = embedder.encode([query_text], convert_to_numpy=True, normalize_embeddings=True)
print(query_vec.shape)

In [ ]:
k = 5   # how many passages to retrieve
scores, indices = index.search(query_vec, k)

print("Scores:", scores)
print("Indices:", indices)

In [ ]:
top_pids = indices[0]

for rank, pid in enumerate(top_pids, start=1):
    print(f"\nRank {rank}, pid {pid}, score {scores[0][rank-1]:.4f}")
    print(corpus[pid])

In [ ]:
def evaluate_hit_k(queries, qrels, index, embedder, k=5, max_queries=200):
    """
    Simple Hit@k evaluation.

    queries: dict[qid] -> {"text": ..., "answers": [...]}
    qrels:   dict[qid] -> list of relevant passage_ids
    index:   FAISS index over passage_embeddings
    embedder: SentenceTransformer model
    k:       top-k to evaluate
    max_queries: evaluate at most this many queries (for speed)
    """
    hits = []
    num_evaluated = 0

    for qid, qinfo in tqdm(queries.items()):
        # skip queries with no labeled relevant passages
        relevant_pids = qrels.get(qid, [])
        if not relevant_pids:
            continue

        query_text = qinfo["text"]

        # 1) embed query
        q_vec = embedder.encode([query_text], convert_to_numpy=True, normalize_embeddings=True)

        # 2) search FAISS
        scores, indices = index.search(q_vec, k)  # indices: shape (1, k)
        retrieved_pids = set(indices[0].tolist())

        # 3) check if any relevant pid is in retrieved
        relevant_set = set(relevant_pids)
        hit = 1 if relevant_set & retrieved_pids else 0
        hits.append(hit)

        num_evaluated += 1
        if num_evaluated >= max_queries:
            break

    if not hits:
        print("No queries with qrels were evaluated.")
        return None

    hit_k = float(np.mean(hits))
    print(f"Evaluated {num_evaluated} queries with labels.")
    print(f"Hit@{k}: {hit_k:.3f}")

    return hit_k


In [ ]:
hit5 = evaluate_hit_k(queries, qrels, index, embedder, k=5, max_queries=200)
hit10 = evaluate_hit_k(queries, qrels, index, embedder, k=10, max_queries=200)


In [ ]:
def retrieve_top_k(query_text, k=5):
    # 1) Embed query
    q_vec = embedder.encode([query_text], convert_to_numpy=True, normalize_embeddings=True)

    # 2) Search FAISS
    scores, indices = index.search(q_vec, k)

    # 3) Turn into a list of (pid, score, passage_text)
    results = []
    for score, pid in zip(scores[0], indices[0]):
        text = corpus[pid]
        results.append({"pid": int(pid), "score": float(score), "text": text})

    return results

In [ ]:
sample_qid = next(iter(queries.keys()))
sample_query = queries[sample_qid]["text"]
top_docs = retrieve_top_k(sample_query, k=5)

top_docs

In [36]:
PROMPT_TEMPLATE = """
You are a question-answering assistant.

You MUST follow these rules:
- Use ONLY the information in the provided passages.
- If the passages do not contain enough information to fully answer, say "The passages do not give enough information to answer this."
- Do NOT use outside knowledge, even if you think you know the answer.
- Do NOT invent facts not supported by the passages.

Passages:
{context}

Question:
{question}

Answer in 2–3 concise, factual sentences.
At the end of your answer, list which passages you used, like: "Sources: [Passage 1], [Passage 3]".

"""

In [37]:
def build_context_block(retrieved_docs):
    parts = []
    for i, doc in enumerate(retrieved_docs, start=1):
        part = f"[Passage {i}] (pid={doc['pid']}, score={doc['score']:.3f})\n{doc['text']}"
        parts.append(part)
    return "\n\n".join(parts)


In [38]:
context = build_context_block(top_docs)
prompt = PROMPT_TEMPLATE.format(context=context, question=sample_query)
print(prompt[:1000])  # sanity check



You are a question-answering assistant.

You MUST follow these rules:
- Use ONLY the information in the provided passages.
- If the passages do not contain enough information to fully answer, say "The passages do not give enough information to answer this."
- Do NOT use outside knowledge, even if you think you know the answer.
- Do NOT invent facts not supported by the passages.

Passages:
[Passage 1] (pid=1, score=0.646)
The Manhattan Project and its atomic bomb helped bring an end to World War II. Its legacy of peaceful uses of atomic energy continues to have an impact on history and science.

[Passage 2] (pid=0, score=0.644)
The presence of communication amid scientific minds was equally important to the success of the Manhattan Project as scientific intellect was. The only cloud hanging over the impressive achievement of the atomic researchers and engineers is what their success truly meant; hundreds of thousands of innocent lives obliterated.

[Passage 3] (pid=3, score=0.634)
The

In [39]:
import requests
import json

OLLAMA_URL = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "llama3.2:3b"  # or whatever you pulled

def call_llm(prompt: str) -> str:
    """
    Call local Ollama model with a simple prompt and return the full response text.
    Uses the /api/generate endpoint with streaming-style output.
    """
    payload = {
        "model": OLLAMA_MODEL,
        "prompt": prompt,
        "stream": True,  # we get chunks line-by-line
    }

    response = requests.post(OLLAMA_URL, json=payload, stream=True)
    response.raise_for_status()

    full_text = []

    for line in response.iter_lines():
        if not line:
            continue
        data = json.loads(line.decode("utf-8"))
        # each chunk has "response" and eventually "done": true
        if "response" in data:
            full_text.append(data["response"])
        if data.get("done"):
            break

    return "".join(full_text).strip()


In [40]:
test_answer = call_llm("Explain what the Manhattan Project was in one short sentence.")
print(test_answer)


The Manhattan Project was a secret research and development project during World War II that aimed to create an atomic bomb using nuclear fission, with a team of scientists led by J. Robert Oppenheimer.


In [41]:
def answer_with_rag(qid, k=5):
    qinfo = queries[qid]
    query_text = qinfo["text"]
    refs = qinfo["answers"]

    retrieved = retrieve_top_k(query_text, k=k)
    context = build_context_block(retrieved)
    prompt = PROMPT_TEMPLATE.format(context=context, question=query_text)

    answer = call_llm(prompt)

    return {
        "qid": qid,
        "query": query_text,
        "references": refs,
        "retrieved": retrieved,
        "answer": answer,
    }

In [34]:
def random_qid_with_labels():
    # only choose queries that actually HAVE relevant passages
    labeled_qids = [qid for qid, rels in qrels.items() if len(rels) > 0]
    return random.choice(labeled_qids)

sample_qid = random_qid_with_labels()
sample_qid, queries[sample_qid]["text"], qrels[sample_qid]

(467556, 'nyu tuition cost', [61])

In [42]:
# some_qid = next(iter(queries.keys()))
# res = answer_with_rag(some_qid, k=10)
res = answer_with_rag(sample_qid, k=10)

print("QID:", res["qid"])
print("Question:", res["query"])
print("\nRAG answer:\n", res["answer"])
print("\nReference answers:\n", res["references"])

print("\nRetrieved passages:\n")
for doc in res["retrieved"]:
    print(f"pid={doc['pid']}, score={doc['score']:.3f}")
    print(doc["text"])
    print("-" * 80)


QID: 467556
Question: nyu tuition cost

RAG answer:
 The reported New York University net price for in-state students is $34268 for the 2013-2014 academic year. This net price includes housing and meal expenses. The total effective in-state tuition, including additional fees, is $46170.

Sources: [Passage 1], [Passage 3]

Reference answers:
 ['$43,746 for the 2014-2015 academic year.']

Retrieved passages:

pid=61, score=0.804
tuition for new york university is $ 43746 for the 2014 2015 academic year this is 73 % more expensive than the national average private non profit four year college tuition of $ 25240he net out of pocket total cost you end up paying or financing though student loans is known as the net price the reported new york university net price for in state students $ 34268 for the 2013 2014 academic year this net price includes housing and meal expenses
--------------------------------------------------------------------------------
pid=68, score=0.785
with this plan you 